# Stage 5: SQRL in its native protocol

Before running: copy the system prompt, user template and `<observation>` format **verbatim** from the
model card into `config/prompts/sqrl.yaml` (`uv run t2sbench fetch-card sqrl-9b` saves the card and prints the
commit sha for `card_revision`). The run stops if SQRL-9B-agentic scores below 60% on BIRD with evidence
(transcripts in `results/stage5/harness_check_transcripts.md`) or if more than 5% of prompts exceed 90% of the
context window.

In [ ]:
from pathlib import Path
import json, os, subprocess
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
from t2sbench.bench import load_results
from t2sbench.report import aggregate
pd.set_option("display.max_colwidth", 120)
RUN = False  # set True to execute the stage from here (it can take hours and cost money)

def sh(cmd):
    print("$", cmd)
    if RUN:
        subprocess.run(cmd, shell=True, check=True)

def summary(stage):
    df = load_results(Path("results"), stage)
    if df.empty:
        print(f"no results for {stage} yet")
        return df
    agg = aggregate(df)
    return agg[["model", "adapter", "strategy", "dataset", "n", "ex", "ci_lo", "ci_hi", "syntax_ok", "no_answer",
                "time_median", "time_p90", "in_tokens_mean", "out_tokens_mean", "cost_per_100"]].sort_values(
                ["dataset", "ex"], ascending=[True, False])

In [ ]:
sh("uv run t2sbench fetch-card sqrl-9b")

In [ ]:
sh("uv run t2sbench stage 5")

In [ ]:
summary("stage5")

In [ ]:
h = Path("results/stage5/harness_check_transcripts.md")
print(h.read_text()[:5000] if h.exists() else "harness check passed or not run")